In [1]:
"""
Plot correlations and metrics per subject

Script for visualizing correlations between HRV and PRV metrics and metrics trend for each subject.

Main functionalities:
- Loads HRV and PRV feature files (.csv) for each subject, for "All Windows" and "Method 2".
- Computes Pearson and Spearman correlation coefficients between HRV and PRV metrics.
- Generates bar plots comparing correlations obtained with the two methods for each subject.
- Plots the temporal evolution of HRV and PRV metrics across windows for "All Windows" and "Method 2".
"""


# =============================================================================
# IMPORT LIBRARIES
# =============================================================================

import os
import re
import pandas as pd

from utils_analysis import compute_correlations_for_metrics

from utils_analysis_plot import (
    plot_correlations_bar,
    plot_metrics
)

%matplotlib qt


# =============================================================================
# VARIABLES DEFINITION
# =============================================================================

# Define the base directory to data
base_path = os.path.join("..", "Data")

# Define paths for the two experimental conditions
methods = {
    "All Windows": os.path.join(base_path, "All Windows"),
    "Method 2": os.path.join(base_path, "Window Removal")
}

# List subjects based on HRV folder of 'All Windows'
subjects_dir = os.path.join(methods["All Windows"], "HRV")
subjects = [f for f in os.listdir(subjects_dir) if f.endswith(".csv")]
subjects = sorted(subjects, key=lambda x: int(re.findall(r'\d+', x)[0]))  # sort numerically

# Assign subjects to groups
groups = ["healthy"] * 10 + ["RBD"] * 10 + ["OSAS"] * 10 + ["PLM"] * 10 + ["mixed"] * 10
subject_to_group = dict(zip(subjects, groups))


# =============================================================================
# LOAD FEATURES AND COMPUTE CORRELATIONS
# =============================================================================

subjects = subjects[:1] # comment this line

for subj_file in subjects:
    subj_name = subj_file.replace(".csv", "")
    group = subject_to_group[subj_file]

    # ------ All Windows ------
    hrv_path = os.path.join(methods["All Windows"], "HRV", f"{subj_name}.csv")
    prv_path = os.path.join(methods["All Windows"], "PRV", f"{subj_name}.csv")

    df_hrv_all = pd.read_csv(hrv_path).drop(columns=["Window", "HRV_RMSSD"], errors="ignore")
    df_prv_all = pd.read_csv(prv_path).drop(columns=["Window", "HRV_RMSSD"], errors="ignore")

    pearson_all, spearman_all = compute_correlations_for_metrics(df_hrv_all, df_prv_all)


    # ------ Method 2 ------
    hrv_path = os.path.join(methods["Method 2"], "HRV", f"{subj_name}.csv")
    prv_path = os.path.join(methods["Method 2"], "PRV", f"{subj_name}.csv")

    df_hrv_m2 = pd.read_csv(hrv_path).drop(columns=["Window", "HRV_RMSSD"], errors="ignore")
    df_prv_m2 = pd.read_csv(prv_path).drop(columns=["Window", "HRV_RMSSD"], errors="ignore")

    pearson_m2, spearman_m2 = compute_correlations_for_metrics(df_hrv_m2, df_prv_m2)


    # =============================================================================
    # BARPLOT OF PEARSON AND SPEARMAN CORRELATIONS
    # =============================================================================

    plot_correlations_bar(pearson_all, spearman_all, 
        pearson_m2, spearman_m2, title=f'{subj_name} - {group}')
    

    # =============================================================================
    # PLOT HRV/PRV METRICS OVER WINDOWS
    # =============================================================================

    plot_metrics(df_hrv_all, df_prv_all, pearson_all, spearman_all, subj_name, group, 'All Windows')
    plot_metrics(df_hrv_m2, df_prv_m2, pearson_m2, spearman_m2, subj_name, group, 'Method 2')
